In [ ]:
import calcium_event_classifier as cec
from calcium_event_classifier.core.dffdataset import DffDataset
from calcium_event_classifier.core.classifier_dff import CalciumEventClassifierDff
from pathlib import Path
import flammkuchen as fl
import torch
import torch.nn as nn
from datetime import datetime
import matplotlib.pyplot as plt

# Set device and seed
device = cec.set_device()
seed = cec.set_seed(1000)

# Calcium Event Classifier (dFF) - Model Training

This notebook trains the CalciumEventClassifierDff model on dFF calcium imaging data. 
It covers:
- dataset loading
- model initialization
- training loop execution
- checkpoint saving
- performance visualization

## 1. Initialize Device and Random Seed

Set up the training environment by:
- Detecting available GPU or defaulting to CPU
- Setting a fixed random seed for reproducibility
- Configuring the loss function (BCEWithLogitsLoss for binary classification)

In [ ]:
# Get today's date for model naming (YYMMDD format)
today = datetime.now().strftime("%y%m%d")
print(f"Training date: {today}")

# Define loss function
criterion = nn.BCEWithLogitsLoss()

## 2. Load Dataset

Load the training dataset from an HDF5 file and create a DffDataset object. The dataset contains:
- **dff**: Delta-F/F calcium imaging traces (ΔF/F₀)
- **label**: Binary labels (0 = no event, 1 = event)

In [ ]:
# Define dataset path
data_path = Path(r"../datasets/251114_dataset.h5")

# Load data from HDF5 file
data = fl.load(data_path)
print(f"✓ Loaded data from: {data_path}")
print(f"  Data keys: {list(data.keys())}")

# Create DffDataset object
dataset = DffDataset(
    data,
    augment=False,  # Set to True to enable data augmentation
    baseline_normalize=True,  # Apply per-trace baseline normalization
)

print(dataset)

## 3. Split Dataset into Train and Validation Sets

Split the dataset into training (75%) and validation (25%) sets using stratified sampling to maintain class balance. Create DataLoaders for batch processing.

In [ ]:
# Define split ratios and batch size
train_fraction = 0.75
validation_fraction = 0.25
batch_size = 16

# Split dataset and create DataLoaders
train_loader, valid_loader = cec.split(
    dataset,
    train_fraction=train_fraction,
    validation_fraction=validation_fraction,
    batch_size=batch_size,
    seed=seed,
    summary=True  # Print dataset statistics
)

## 4. Initialize Model Architecture

Create a CalciumEventClassifierDff model with specified architecture hyperparameters:
- **Input**: 1 channel (dFF only)
- **Convolutional layers**: 3 layers with progressive channel expansion
- **Regularization**: Dropout and LeakyReLU activation
- **Output**: Binary classification (sigmoid activation applied during inference)

In [ ]:
# Define architecture hyperparameters
input_channels = 1  # dFF only (single channel)
conv1_channels = 16
conv2_channels = 32
conv3_channels = 64
conv1_kernel = 5
conv2_kernel = 3
conv3_kernel = 2
leaky_relu_negative_slope = 0.05
dropout_rate = 0.1
pool_kernel = 2

print("=== Model Architecture Hyperparameters ===")
print(f"Input channels:           {input_channels}")
print(f"Conv1 channels:           {conv1_channels}, kernel: {conv1_kernel}")
print(f"Conv2 channels:           {conv2_channels}, kernel: {conv2_kernel}")
print(f"Conv3 channels:           {conv3_channels}, kernel: {conv3_kernel}")
print(f"LeakyReLU negative slope: {leaky_relu_negative_slope}")
print(f"Dropout rate:             {dropout_rate}")
print(f"Pool kernel:              {pool_kernel}")

# Initialize classifier
classifier = CalciumEventClassifierDff(
    trace_length =              len(dataset[0][0][0]),  # Get trace length from first sample
    input_channels =            input_channels,
    conv1_channels =            conv1_channels,
    conv2_channels =            conv2_channels,
    conv3_channels =            conv3_channels,
    conv1_kernel =              conv1_kernel,
    conv2_kernel =              conv2_kernel,
    conv3_kernel =              conv3_kernel,
    leaky_relu_negative_slope = leaky_relu_negative_slope,
    dropout_rate =              dropout_rate,
    pool_kernel =               pool_kernel,
).to(device)

print(f"\n✓ Model initialized on {device}")
print(f"  Trace length: {len(dataset[0][0][0])}")

## 5. Configure Training Hyperparameters

Set up all training parameters including:
- **Epochs**: Maximum number of training iterations
- **Learning rate**: Initial learning rate with decay schedule
- **Regularization**: L1 and L2 penalty coefficients
- **Early stopping**: Patience for validation metric monitoring

In [ ]:
# Define training hyperparameters
epochs = 500
learning_rate = 1e-5
lr_drop_factor = 0.5  # Factor to reduce LR by
lr_drop_patience = 5  # Epochs before LR reduction
lambda1 = 1e-5  # L1 regularization
lambda2 = 1e-4  # L2 regularization
patience = 8  # Early stopping patience

print("=== Training Hyperparameters ===")
print(f"Epochs:              {epochs}")
print(f"Learning rate:       {learning_rate}")
print(f"LR drop factor:      {lr_drop_factor}")
print(f"LR drop patience:    {lr_drop_patience}")
print(f"L1 regularization:   {lambda1}")
print(f"L2 regularization:   {lambda2}")
print(f"Early stop patience: {patience}")

## 6. Train the Model

Execute the training loop. This will:
- Iterate through training/validation batches
- Update model weights using backpropagation
- Track loss and metrics on both sets
- Apply learning rate scheduling and early stopping
- Return the best model and training history

In [ ]:
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60 + "\n")

# Train the model
(
    model,
    train_loss,
    validation_loss,
    train_f1,
    validation_f1,
    validation_precision,
    validation_recall,
    best_thresholds,
    validation_auc_pr,
    valid_epoch_features,
    valid_epoch_labels,
    train_epoch_features,
    train_epoch_labels,
) = cec.train(
    train_loader=train_loader,
    valid_loader=valid_loader,
    model=classifier,
    criterion=criterion,
    device=device,
    epochs=epochs,
    learning_rate=learning_rate,
    lr_drop_factor=lr_drop_factor,
    lr_drop_patience=lr_drop_patience,
    lambda1=lambda1,
    lambda2=lambda2,
    patience=patience,
)

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)

## 7. Save Model Checkpoint

Save the trained model, hyperparameters, and training metrics to a checkpoint file. The checkpoint includes:
- Model state dictionary (weights and biases)
- All training hyperparameters for reproducibility
- Training/validation metrics history
- Latent features for analysis

In [ ]:
# Create checkpoint dictionary
checkpoint = {
    "model_state_dict": model.state_dict(),
    "training_dataset": data_path,
    "train_loss": train_loss,
    "validation_loss": validation_loss,
    "train_f1": train_f1,
    "validation_f1": validation_f1,
    "validation_precision": validation_precision,
    "validation_recall": validation_recall,
    "best_thresholds": best_thresholds,
    "validation_auc_pr": validation_auc_pr,
    "valid_epoch_features": valid_epoch_features,
    "valid_epoch_labels": valid_epoch_labels,
    "train_epoch_features": train_epoch_features,
    "train_epoch_labels": train_epoch_labels,
    "hyperparams": {
        "learning_rate": learning_rate,
        "lr_drop_factor": lr_drop_factor,
        "lr_drop_patience": lr_drop_patience,
        "lambda1": lambda1,
        "lambda2": lambda2,
        "epochs": epochs,
        "patience": patience,
        "input_channels": input_channels,
        "conv1_channels": conv1_channels,
        "conv2_channels": conv2_channels,
        "conv3_channels": conv3_channels,
        "conv1_kernel": conv1_kernel,
        "conv2_kernel": conv2_kernel,
        "conv3_kernel": conv3_kernel,
        "pool_kernel": pool_kernel,
        "dropout": dropout_rate,
        "leaky_relu_negative_slope": leaky_relu_negative_slope,
        "batch_size": batch_size,
        "random_seed": seed,
        "use_dff_only": True,
    }
}

# Define save path (YYMMDD_model_dff.pth)
save_path = Path(rf"models/{today}_model_dff.pth")

# Save checkpoint
torch.save(checkpoint, save_path)
print(f"\n✓ Model checkpoint saved to: {save_path}")
print(f"  File size: {save_path.stat().st_size / 1e6:.1f} MB")

## 8. Visualize Training Curves

Plot the training and validation loss and F1 score over epochs to visualize model convergence and identify overfitting/underfitting.

In [ ]:
# Create training curves plot
fig, ax = plt.subplots(figsize=(12, 6))

# Plot loss and F1 scores
ax.plot(train_loss, linewidth=2, label="Train Loss")
ax.plot(validation_loss, linewidth=2, label="Validation Loss")
ax.plot(train_f1, linewidth=2, label="Train F1")
ax.plot(validation_f1, linewidth=2, label="Validation F1")

# Formatting
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss / F1 Score", fontsize=12)
ax.set_title("Training and Validation Metrics (dFF Only)", fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc="best")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
print("\n=== Final Training Metrics ===")
print(f"Best epoch: {len(train_loss)}")
print(f"Final train loss:       {train_loss[-1]:.4f}")
print(f"Final validation loss:  {validation_loss[-1]:.4f}")
print(f"Final train F1:         {train_f1[-1]:.4f}")
print(f"Final validation F1:    {validation_f1[-1]:.4f}")
print(f"Best validation F1:     {max(validation_f1):.4f}")
print(f"Best validation PR-AUC: {max(validation_auc_pr):.4f}")